In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('boat_data.csv')

print('Libraries imported.')
print('Dataset loaded.')
print(f'Shape: {df.shape[0]} rows, {df.shape[1]} columns')


Libraries imported.
Dataset loaded.
Shape: 9888 rows, 10 columns


In [4]:
print('=== Shape ===')
print(f'{df.shape[0]} rows and {df.shape[1]} columns')

print('\n=== Column Names ===')
print(df.columns.tolist())

print('\n=== Data Types ===')
print(df.dtypes)

print('\n1st 5 row')
df.head()

print('\n=== Summary Statistics ===')
df.describe()

print('\n=== Missing Value Summary ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])


=== Shape ===
9888 rows and 10 columns

=== Column Names ===
['Price', 'Boat Type', 'Manufacturer', 'Type', 'Year Built', 'Length', 'Width', 'Material', 'Location', 'Number of views last 7 days']

=== Data Types ===
Price                              str
Boat Type                          str
Manufacturer                       str
Type                               str
Year Built                       int64
Length                         float64
Width                          float64
Material                           str
Location                           str
Number of views last 7 days      int64
dtype: object

1st 5 row

=== Summary Statistics ===

=== Missing Value Summary ===
              Missing Count  Missing %
Manufacturer           1338      13.53
Type                      6       0.06
Length                    9       0.09
Width                    56       0.57
Material               1749      17.69
Location                 36       0.36


In [ ]:
print('\n=== Shape ===')
print(f'{df.shape[0]} rows and {df.shape[1]} columns')

print('\n=== Column Names ===')
print(df.columns.tolist())

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== 1st 5 row ===')
df.head()



In [ ]:
# Sec 5:
# --- Feature 1: Boat Age ---
# Derived from Year Built. Older boats generally depreciate more.
df_clean['Boat Age'] = 2024 - df_clean['Year Built'].astype(int)
print('Feature 1 — Boat Age created.')
print(df_clean['Boat Age'].describe())

# --- Feature 2: Price per Meter ---
# Shows value density — is the boat expensive for its size?
df_clean['Price per Meter'] = (df_clean['Price_USD'] / df_clean['Length']).round(2)
print('Feature 2 — Price per Meter created.')
print(df_clean['Price per Meter'].describe())

# --- Feature 3: Boat Category ---
# Groups Boat Type into 3 broader categories for easier comparison
sailboat_types = ['Sailboat', 'Sailing yacht', 'Catamaran']
powerboat_types = ['Motor yacht', 'Motorboat', 'Sport boat', 'Cabin boat', 'Bowrider']

def categorize_boat(boat_type):
    if pd.isnull(boat_type):
        return 'Other'
    for s in sailboat_types:
        if s.lower() in str(boat_type).lower():
            return 'Sailboat'
    for p in powerboat_types:
        if p.lower() in str(boat_type).lower():
            return 'Powerboat'
    return 'Other'

df_clean['Boat Category'] = df_clean['Boat Type'].apply(categorize_boat)
print('Feature 3 — Boat Category created.')
print(df_clean['Boat Category'].value_counts())

In [ ]:
## --- Chart 1: Average Price by Boat Category (Bar Chart) ---
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
cat_price.plot(kind='bar', ax=ax, color=colors, edgecolor='black', width=0.6)
ax.set_title('Average Listing Price by Boat Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Boat Category', fontsize=12)
ax.set_ylabel('Average Price (USD)', fontsize=12)
ax.set_xticklabels(cat_price.index, rotation=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('chart1_price_by_category.png', dpi=150)
plt.show()
print('CHART 1 INTERPRETATION:')
print('This bar chart compares the average listing price across the three boat categories.')
print('It directly answers Question 1: whether sailboats or powerboats command higher prices.')
print('The taller the bar, the more expensive boats in that category tend to be on average.') 

In [ ]:

# --- Chart 2: Length vs Price (Scatter Plot) ---
fig, ax = plt.subplots(figsize=(9, 6))
colors_map = {'Sailboat': '#1f77b4', 'Powerboat': '#ff7f0e', 'Other': '#aaaaaa'}
for category, group in df_clean[df_clean['Price_USD'] < 2_000_000].groupby('Boat Category'):
    ax.scatter(group['Length'], group['Price_USD'],
               label=category, alpha=0.4, s=15,
               color=colors_map.get(category, '#888888'))
ax.set_title('Boat Length vs Listing Price', fontsize=14, fontweight='bold')
ax.set_xlabel('Length (meters)', fontsize=12)
ax.set_ylabel('Price (USD)', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(title='Boat Category')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('chart2_length_vs_price.png', dpi=150)
plt.show()
print('CHART 2 INTERPRETATION:')
print('This scatter plot shows the relationship between boat length and price.')
print('An upward trend would confirm that larger boats are priced significantly higher.')
print('The color coding reveals whether this trend holds differently for sailboats vs powerboats.')

In [ ]:
# --- Chart 3: Boat Age Distribution (Histogram) ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df_clean['Boat Age'].clip(0, 100), bins=40, color='#2ca02c', edgecolor='white', alpha=0.8)
ax.set_title('Distribution of Boat Age', fontsize=14, fontweight='bold')
ax.set_xlabel('Boat Age (years)', fontsize=12)
ax.set_ylabel('Number of Listings', fontsize=12)
ax.axvline(df_clean['Boat Age'].median(), color='red', linestyle='--', linewidth=1.5, label=f'Median age: {df_clean["Boat Age"].median():.0f} yrs')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('chart3_boat_age_distribution.png', dpi=150)
plt.show()
print('CHART 3 INTERPRETATION:')
print('This histogram shows how old the listed boats are.')
print('A right-skewed distribution would indicate that most boats listed are relatively young.')
print('The red median line gives a quick reference for what a typical listing age looks like.')

In [ ]:
# --- Chart 4: Price Distribution by Boat Category (Box Plot) ---
fig, ax = plt.subplots(figsize=(9, 6))
categories = ['Sailboat', 'Powerboat', 'Other']
data_to_plot = [df_clean[df_clean['Boat Category'] == cat]['Price_USD'].clip(upper=1_500_000).dropna().values
                for cat in categories]
bp = ax.boxplot(data_to_plot, labels=categories, patch_artist=True, notch=False)
box_colors = ['#1f77b4', '#ff7f0e', '#aaaaaa']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_title('Price Distribution by Boat Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Boat Category', fontsize=12)
ax.set_ylabel('Price (USD)', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('chart4_price_boxplot.png', dpi=150)
plt.show()
print('CHART 4 INTERPRETATION:')
print('This box plot shows the spread of prices within each boat category.')
print('The box shows where the middle 50% of prices fall; the line inside is the median.')
print('Dots outside the whiskers are outlier listings — unusually expensive or cheap boats.')

In [ ]:
print('='*60)
print('KEY FINDINGS')
print('='*60)

print(f'''
Finding 1 — Price by category:
  {cat_price.idxmax()} boats have the highest average listing price
  at ${cat_price.max():,.0f}, compared to ${cat_price.min():,.0f} for
  {cat_price.idxmin()} boats. This suggests significant price differences
  between boat types on the market.

Finding 2 — Length and price relationship:
  The Pearson correlation between length and price is {correlation:.2f},
  indicating a {'strong' if abs(correlation) > 0.5 else 'moderate'} positive
  relationship. Longer boats consistently command higher prices.

Finding 3 — Outlier listings:
  {len(outliers)} boat listings were identified as statistical outliers
  with z-scores above 3. These extreme prices likely represent luxury yachts
  or data entry errors and should be excluded from general analysis.

Finding 4 — Geographic pricing differences:
  The top-priced country averages significantly higher than the dataset
  overall, suggesting that some markets list premium boats more frequently.

Finding 5 — Buyer interest (views):
  {views_by_cat.idxmax()} boats receive the most average views per week,
  at {views_by_cat.max():.1f} views. This signals stronger buyer demand
  for this category in the current market.
''')

print('='*60)
print('LIMITATIONS')
print('='*60)
print('''
1. Currency conversion used approximate historical exchange rates,
   which may not reflect exact values at the time of listing.

2. Missing values in Material and Width were handled with fill strategies
   that may introduce small biases into group comparisons.

3. The dataset only reflects listed prices, not actual sale prices,
   so true market value may differ.

4. Correlation does not imply causation — length correlates with price
   but other factors (brand, condition, age) also play major roles.

5. The boat category classification is rule-based and may misclassify
   niche boat types not matching the defined keywords.
''')

print('='*60)
print('CONCLUSION')
print('='*60)
print('''
This analysis of the Boat Sales dataset revealed meaningful patterns
in how boat type, size, and geography influence listing prices and
buyer interest. Boat length is a reliable predictor of price, with
a positive correlation confirmed through statistical analysis.
Significant price outliers exist and were detected using NumPy z-scores.
Future work could include normalizing prices for boat condition and
incorporating actual sale data to better reflect true market dynamics.
''')